# 1. SPARK GRAPH-X
- Môn học: Big Data - CO3137
- Ngày 14/04/2026
- Lớp: L01

| STT | Họ tên | MSSV |
| :---: | :--- | :---: |
| 1 | Lê Đình Đức | 2310774 |
| 2 | Nguyễn Văn Công Thành | 231xxx |

# 3. Exercise

### Exercise 0: Prepare movie data
- Read data from 3 Kafka topics:
  - Movies (`Lab1_movies`)
  - Ratings (`Lab1_ratings`)
  - Tags (`Lab1_tags`)
- Define schemas for movies, ratings, and tags.
- Convert Kafka JSON values to DataFrames using the corresponding schemas.


In [1]:
import time

import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import abs as spark_abs, sum as spark_sum
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

spark = (
    SparkSession.builder
    .appName("Lab3")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "io.graphframes:graphframes-spark4_2.13:0.9.3,"
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
        "org.apache.kafka:kafka-clients:3.9.1"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir("file:///tmp/lab3-graphframes-checkpoint")

KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"


/mnt/d/myBK/HK252/BigData/BDNotebook/bdnotebook/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 11:46:10 WARN Utils: Your hostname, DESKTOP, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 11:46:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/d/myBK/HK252/BigData/BDNotebook/bdnotebook/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added a

In [2]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

source_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
source_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
source_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

print("Ratings preview:")
source_ratings.show(3, truncate=False)
print("Movies preview:")
source_movies.show(3, truncate=False)
print("Tags preview:")
source_tags.show(3, truncate=False)


Ratings preview:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
+------+-------+------+---------+
only showing top 3 rows
Movies preview:
+-------+-----------------------+-------------------------------------------+
|movieId|title                  |genres                                     |
+-------+-----------------------+-------------------------------------------+
|1      |Toy Story (1995)       |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)         |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)|Comedy|Romance                             |
+-------+-----------------------+-------------------------------------------+
only showing top 3 rows
Tags preview:
+------+-------+---------------+----------+
|userId|movieId|tag            |timestamp |
+------+-------+---------------+-

In [3]:
TOPIC_MOVIES = "Lab1_movies"
TOPIC_RATINGS = "Lab1_ratings"
TOPIC_TAGS = "Lab1_tags"
TOPICS = [TOPIC_MOVIES, TOPIC_RATINGS, TOPIC_TAGS]

admin_client = AdminClient({"bootstrap.servers": KAFKA_BROKERS})

# Reset topics so each notebook run prepares a clean copy of the MovieLens data.
delete_futures = admin_client.delete_topics(TOPICS, operation_timeout=10)
for topic, future in delete_futures.items():
    try:
        future.result()
        print(f"Deleted topic: {topic}")
    except Exception as e:
        print(f"Topic {topic} was not deleted or did not exist: {e}")

time.sleep(5)

new_topics = [NewTopic(topic=topic, num_partitions=3, replication_factor=1) for topic in TOPICS]
create_futures = admin_client.create_topics(new_topics)
for topic, future in create_futures.items():
    try:
        future.result()
        print(f"Created topic: {topic}")
    except Exception as e:
        print(f"Topic {topic} was not created or already exists: {e}")


Deleted topic: Lab1_movies
Deleted topic: Lab1_ratings
Deleted topic: Lab1_tags
Created topic: Lab1_movies
Created topic: Lab1_ratings
Created topic: Lab1_tags


In [4]:
source_movies.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", TOPIC_MOVIES) \
    .save()

source_ratings.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", TOPIC_RATINGS) \
    .save()

source_tags.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", TOPIC_TAGS) \
    .save()

print("Successfully prepared MovieLens data in Kafka topics:")
for topic in TOPICS:
    print(f"- {topic}")


Successfully prepared MovieLens data in Kafka topics:
- Lab1_movies
- Lab1_ratings
- Lab1_tags


In [5]:
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True),
])


In [6]:
def read_kafka_topic(topic_name, schema):
    kafka_df = (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKERS)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .option("endingOffsets", "latest")
        .load()
    )

    return (
        kafka_df
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), schema).alias("data"))
        .select("data.*")
    )

movies = read_kafka_topic(TOPIC_MOVIES, movie_schema).dropDuplicates(["movieId"])
ratings = read_kafka_topic(TOPIC_RATINGS, rating_schema).dropDuplicates(["userId", "movieId"])
tags = read_kafka_topic(TOPIC_TAGS, tag_schema)

print(f"Movies: {movies.count()} rows")
print(f"Ratings: {ratings.count()} rows")
print(f"Tags: {tags.count()} rows")

movies.printSchema()
ratings.printSchema()
tags.printSchema()


Movies: 9742 rows


Ratings: 100836 rows
Tags: 3683 rows
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)



In [9]:
import numpy as np
from graphframes import GraphFrame

user_vertices = (
    ratings
    .select(col("userId"))
    .distinct()
    .withColumn("id", concat(lit("u_"), col("userId")))
    .withColumn("type", lit("user"))
    .select("id", "type", "userId")
)

movie_vertices = (
    movies
    .withColumn("id", concat(lit("m_"), col("movieId")))
    .withColumn("type", lit("movie"))
    .select("id", "type", "movieId", "title", "genres")
)

vertices = user_vertices.unionByName(movie_vertices, allowMissingColumns=True)

edges = (
    ratings
    .withColumn("src", concat(lit("u_"), col("userId")))
    .withColumn("dst", concat(lit("m_"), col("movieId")))
    .select("src", "dst", "userId", "movieId", col("rating").alias("weight"), "timestamp")
)

g = GraphFrame(vertices, edges)

print(f"Vertices: {g.vertices.count()} rows")
print(f"Edges: {g.edges.count()} rows")


Vertices: 10352 rows


[Stage 51:>                                                         (0 + 3) / 3]

Edges: 100836 rows


### Exercise 1: Check for popularity bias
- Compute the in-degree of movies (number of distinct raters).
- Compute weighted in-degree (sum of rating weights).
- Output top-20 movies by in-degree and weighted in-degree.


In [10]:
from pyspark.sql.functions import countDistinct, desc
from pyspark.sql.functions import sum as spark_sum

movie_popularity = (
    ratings
    .groupBy("movieId")
    .agg(
        countDistinct("userId").alias("inDegree"),
        spark_sum("rating").alias("weightedInDegree")
    )
    .join(movies, on="movieId", how="inner")
    .select("movieId", "title", "genres", "inDegree", "weightedInDegree")
)

top20_by_indegree = movie_popularity.orderBy(desc("inDegree"), desc("weightedInDegree")).limit(20)
top20_by_weighted_indegree = movie_popularity.orderBy(desc("weightedInDegree"), desc("inDegree")).limit(20)

print("Top 20 movies by in-degree:")
top20_by_indegree.show(20, truncate=False)

print("Top 20 movies by weighted in-degree:")
top20_by_weighted_indegree.show(20, truncate=False)


Top 20 movies by in-degree:


+-------+------------------------------------------------------------------------------+-------------------------------------------+--------+----------------+
|movieId|title                                                                         |genres                                     |inDegree|weightedInDegree|
+-------+------------------------------------------------------------------------------+-------------------------------------------+--------+----------------+
|356    |Forrest Gump (1994)                                                           |Comedy|Drama|Romance|War                   |329     |1370.0          |
|318    |Shawshank Redemption, The (1994)                                              |Crime|Drama                                |317     |1404.0          |
|296    |Pulp Fiction (1994)                                                           |Comedy|Crime|Drama|Thriller                |307     |1288.5          |
|593    |Silence of the Lambs, The (1991)     

+-------+------------------------------------------------------------------------------+-------------------------------------------+--------+----------------+
|movieId|title                                                                         |genres                                     |inDegree|weightedInDegree|
+-------+------------------------------------------------------------------------------+-------------------------------------------+--------+----------------+
|318    |Shawshank Redemption, The (1994)                                              |Crime|Drama                                |317     |1404.0          |
|356    |Forrest Gump (1994)                                                           |Comedy|Drama|Romance|War                   |329     |1370.0          |
|296    |Pulp Fiction (1994)                                                           |Comedy|Crime|Drama|Thriller                |307     |1288.5          |
|2571   |Matrix, The (1999)                   

**Insights:**
1. Movies appearing near the top of both rankings show popularity bias because a large audience also gives them a very large total rating mass.
2. Differences between the two rankings reveal movies whose popularity is not matched by equally strong rating weight, or whose smaller audience gives higher scores.
3. Weighted in-degree should not be interpreted as pure quality because it rewards the number of raters and can amplify already-popular movies.


### Exercise 2: Get the 20 most relevant movies using PageRank
- Run global PageRank on the user -> movie graph.
- Output top-20 movies by global PageRank with genres and PageRank score.


In [11]:
pagerank_result = g.pageRank(resetProbability=0.15, maxIter=10)

top20_pagerank_movies = (
    pagerank_result.vertices
    .filter(col("type") == "movie")
    .select("movieId", "title", "genres", col("pagerank").alias("pageRankScore"))
    .orderBy(desc("pageRankScore"))
    .limit(20)
)

top20_pagerank_movies.show(20, truncate=False)


+-------+-----------------------------------------+-------------------------------------------+------------------+
|movieId|title                                    |genres                                     |pageRankScore     |
+-------+-----------------------------------------+-------------------------------------------+------------------+
|318    |Shawshank Redemption, The (1994)         |Crime|Drama                                |4.544090131790101 |
|356    |Forrest Gump (1994)                      |Comedy|Drama|Romance|War                   |4.227459071298885 |
|296    |Pulp Fiction (1994)                      |Comedy|Crime|Drama|Thriller                |4.022161161548665 |
|593    |Silence of the Lambs, The (1991)         |Crime|Horror|Thriller                      |3.793749529100372 |
|2571   |Matrix, The (1999)                       |Action|Sci-Fi|Thriller                     |3.5484553443912965|
|110    |Braveheart (1995)                        |Action|Drama|War             

### Exercise 3 (bonus): Motifs for polarization (disagreement)
- Find movies where two users rated the same movie with an absolute rating difference of at least 3.0.
- Rank movies by polarized pair count.


In [12]:
polarized_motifs = (
    g.find("(u1)-[r1]->(m); (u2)-[r2]->(m)")
    .filter(col("u1.type") == "user")
    .filter(col("u2.type") == "user")
    .filter(col("m.type") == "movie")
    .filter(col("u1.id") < col("u2.id"))
    .filter(spark_abs(col("r1.weight") - col("r2.weight")) >= 3.0)
)

top10_polarizing_movies = (
    polarized_motifs
    .groupBy(col("m.movieId").alias("movieId"), col("m.title").alias("title"))
    .count()
    .withColumnRenamed("count", "polarized_pairs")
    .orderBy(desc("polarized_pairs"))
    .limit(10)
)

top10_polarizing_movies.show(10, truncate=False)


[Stage 455:>                                                        (0 + 1) / 1]

+-------+---------------------------------------------------------+---------------+
|movieId|title                                                    |polarized_pairs|
+-------+---------------------------------------------------------+---------------+
|296    |Pulp Fiction (1994)                                      |3083           |
|2571   |Matrix, The (1999)                                       |2633           |
|527    |Schindler's List (1993)                                  |1747           |
|356    |Forrest Gump (1994)                                      |1569           |
|110    |Braveheart (1995)                                        |1555           |
|2858   |American Beauty (1999)                                   |1499           |
|260    |Star Wars: Episode IV - A New Hope (1977)                |1485           |
|593    |Silence of the Lambs, The (1991)                         |1402           |
|4993   |Lord of the Rings: The Fellowship of the Ring, The (2001)|1178     

**Insights:**
1. Highly polarizing movies usually have enough viewers to create many opposing user pairs, so controversy and popularity often appear together.
2. A large polarized-pair count means the average rating hides disagreement and should be interpreted together with rating dispersion.
3. Recommender systems should treat these movies carefully because the same title can be loved by one user group and strongly disliked by another.
